# 🩺 Wellness Data Integrity & Governance Engine

> **An end-to-end data quality and governance pipeline** for a longevity & wellness platform.
> Extracts raw clinical and wearable telemetry, sanitizes it with statistical methods,
> and serves it through interactive dashboards.

**Run this entire notebook end-to-end in Google Colab** — no API keys, no downloads, no Docker required.

---

### What this notebook demonstrates

| Phase | Responsibility | Section |
|---|---|---|
| 0 | Environment setup | Setup |
| 1 | Extract data from databases (Redshift, MongoDB) | Phase 1 |
| 2 | Implement a data governance framework + documentation | Phase 2 |
| 3 | Track quality; automated checks; statistical anomaly detection | Phase 3 |
| 4 | Identify and resolve root causes (cross-functional collab) | Phase 4 |
| 5 | Customized reports and dashboards | Phase 5 |

### Why this design

A real wellness platform mixes **structured clinical data** (warehoused in Redshift/Postgres) with **semi-structured wearable telemetry** (stored in MongoDB). We simulate both stacks inside this notebook:

- **SQLite** (Python stdlib) plays the role of Postgres/Redshift — same SQL dialect
- **TinyDB** plays the role of MongoDB — same document-store semantics

The same code structure works against the real enterprise stack when you swap the connectors.


---
## 🔧 Setup — install dependencies and configure logging


In [ ]:
# Install lightweight Colab dependencies (TinyDB + Faker)
# pandas / numpy / scipy / scikit-learn / plotly are pre-installed in Colab.
%pip install -q tinydb faker

# Standard library
import json
import logging
import sqlite3
import warnings
from datetime import datetime, timedelta, timezone
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
from faker import Faker
from scipy import stats
from sklearn.impute import KNNImputer, SimpleImputer
from tinydb import TinyDB, Query
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("wellness")

# Reproducibility
SEED = 42
np.random.seed(SEED)
Faker.seed(SEED)
fake = Faker()

# Workspace
WORKDIR = Path("./wellness_workspace")
WORKDIR.mkdir(exist_ok=True)
SQLITE_PATH = WORKDIR / "clinical.db"
TINYDB_PATH = WORKDIR / "telemetry.json"

print("✅ Environment ready")
print(f"   workdir: {WORKDIR.resolve()}")
print(f"   SQLite : {SQLITE_PATH.name}")
print(f"   TinyDB : {TINYDB_PATH.name}")


---
## 🟦 Phase 1 — Data Ingestion

We generate two realistic datasets that mirror real public health releases:

| Simulated dataset | Real-world analogue | Stored in |
|---|---|---|
| **`clinical`** schema (demographics, cardiovascular, lab results, body measurements) | NHANES (CDC) | SQLite |
| **`telemetry`** collections (user_profiles, daily_activity, sleep_records, heart_rate) | Fitbit Fitness Tracker (Kaggle) | TinyDB |

We **intentionally inject errors** — negative heart rates, impossibly old ages, duplicate sleep records, missing values, and statistical outliers — so the quality engine has something real to catch.

> **Swap in real data:** drop the NHANES CSV files into `wellness_workspace/raw/` and the cells below will pick them up automatically.


### 1a. Generate NHANES-style clinical data → SQLite (Postgres analogue)


In [ ]:
# ---------------------------------------------------------------------------
# Generate a realistic-looking NHANES-style clinical dataset.
# Schemas are aligned with the real NHANES release so swap-in is trivial.
# ---------------------------------------------------------------------------
N_RESPONDENTS = 2_500

rng = np.random.default_rng(SEED)

# ---- Demographics ---------------------------------------------------------
demographics = pd.DataFrame({
    "seqn":             range(100_000, 100_000 + N_RESPONDENTS),
    "gender":           rng.choice([1, 2], size=N_RESPONDENTS, p=[0.49, 0.51]),
    "age_years":        rng.integers(18, 85, size=N_RESPONDENTS),
    "race_ethnicity":   rng.choice([1, 2, 3, 4, 6, 7], size=N_RESPONDENTS,
                                   p=[0.15, 0.10, 0.45, 0.20, 0.07, 0.03]),
    "education_level":  rng.choice([1, 2, 3, 4, 5], size=N_RESPONDENTS,
                                   p=[0.05, 0.10, 0.30, 0.30, 0.25]),
    "income_to_poverty":np.round(rng.uniform(0.3, 5.0, N_RESPONDENTS), 2),
    "survey_cycle":     "2017-2018",
})

# Inject a few dirty rows
demographics.loc[7,  "age_years"] = 150       # impossible age
demographics.loc[42, "age_years"] = -3        # impossible age
demographics.loc[100, "income_to_poverty"] = np.nan

# ---- Cardiovascular -------------------------------------------------------
cardio = pd.DataFrame({
    "seqn":           demographics["seqn"],
    "exam_date":      pd.to_datetime("2018-01-01") +
                      pd.to_timedelta(rng.integers(0, 365, N_RESPONDENTS), unit="D"),
    "systolic_bp":    np.round(rng.normal(122, 16, N_RESPONDENTS), 1),
    "diastolic_bp":   np.round(rng.normal(78, 11, N_RESPONDENTS), 1),
    "pulse_rate_bpm": np.round(rng.normal(72, 11, N_RESPONDENTS), 1),
    "pulse_regular":  rng.choice([True, False], N_RESPONDENTS, p=[0.95, 0.05]),
})

# Inject statistical outliers AND impossible values
cardio.loc[5,  "systolic_bp"]    = 999            # device error
cardio.loc[15, "systolic_bp"]    = -50            # impossible
cardio.loc[25, "pulse_rate_bpm"] = 25             # bradycardia / device error
cardio.loc[35, "diastolic_bp"]   = 200            # impossible
cardio.loc[45, "systolic_bp"]    = cardio.loc[45, "diastolic_bp"] - 10  # logically inconsistent
cardio.loc[200:210, "pulse_rate_bpm"] = np.nan   # missing block

# ---- Lab results ----------------------------------------------------------
labs = pd.DataFrame({
    "seqn":              demographics["seqn"],
    "sample_date":       cardio["exam_date"],
    "total_cholesterol": np.round(rng.normal(195, 35, N_RESPONDENTS), 1),
    "hdl_cholesterol":   np.round(rng.normal(55, 15, N_RESPONDENTS), 1),
    "ldl_cholesterol":   np.round(rng.normal(115, 30, N_RESPONDENTS), 1),
    "triglycerides":     np.round(rng.normal(140, 60, N_RESPONDENTS), 1),
    "fasting_glucose":   np.round(rng.normal(100, 20, N_RESPONDENTS), 1),
    "hba1c":             np.round(rng.normal(5.5, 0.8, N_RESPONDENTS), 2),
})

labs.loc[60, "fasting_glucose"] = -999            # impossible
labs.loc[70, "hba1c"]           = 35              # impossible
labs.loc[80:90, "ldl_cholesterol"] = np.nan        # missing block

# ---- Body measurements ----------------------------------------------------
height = rng.normal(168, 10, N_RESPONDENTS)
weight = rng.normal(78, 17, N_RESPONDENTS)
body = pd.DataFrame({
    "seqn":         demographics["seqn"],
    "measure_date": cardio["exam_date"],
    "height_cm":    np.round(height, 1),
    "weight_kg":    np.round(weight, 1),
    "bmi":          np.round(weight / (height / 100) ** 2, 1),
    "waist_cm":     np.round(rng.normal(92, 14, N_RESPONDENTS), 1),
})
body.loc[150, "bmi"] = 8                          # impossible

# ---- Load into SQLite (Redshift analogue) ---------------------------------
with sqlite3.connect(SQLITE_PATH) as conn:
    demographics.to_sql("demographics",      conn, index=False, if_exists="replace")
    cardio.to_sql      ("cardiovascular",    conn, index=False, if_exists="replace")
    labs.to_sql        ("lab_results",       conn, index=False, if_exists="replace")
    body.to_sql        ("body_measurements", conn, index=False, if_exists="replace")
    # Audit / quality log table
    conn.execute('''
        CREATE TABLE IF NOT EXISTS data_quality_log (
            log_id        INTEGER PRIMARY KEY AUTOINCREMENT,
            check_name    TEXT, table_name TEXT, column_name TEXT,
            record_id     TEXT, severity TEXT, description TEXT,
            flagged_value TEXT, detected_at TEXT
        )''')

log.info("Loaded %d demographics rows into SQLite", len(demographics))
log.info("Loaded %d cardiovascular rows into SQLite", len(cardio))
log.info("Loaded %d lab results into SQLite", len(labs))
log.info("Loaded %d body measurements into SQLite", len(body))
demographics.head()


### 1b. Generate Fitbit-style wearable telemetry → TinyDB (MongoDB analogue)


In [ ]:
# ---------------------------------------------------------------------------
# Generate 60 users × 90 days of wearable telemetry.
# ---------------------------------------------------------------------------
N_USERS = 60
N_DAYS  = 90
start_date = datetime(2024, 1, 1, tzinfo=timezone.utc)

# Build user profiles (with PII so we have something to mask in Phase 2)
user_profiles = []
for i in range(N_USERS):
    user_profiles.append({
        "user_id":      f"u_{i:04d}",
        "full_name":    fake.name(),                       # PII
        "email":        fake.email(),                      # PII
        "phone":        fake.phone_number(),               # PII
        "device_id":    fake.uuid4(),                      # PII
        "age":          int(rng.integers(20, 80)),
        "gender":       rng.choice(["M", "F", "NB"]),
        "dietary_pref": rng.choice(["omnivore", "vegetarian", "vegan",
                                    "keto", "mediterranean"]),
        "created_at":   (start_date - timedelta(days=int(rng.integers(30, 720)))).isoformat(),
    })

# Inject one user with impossible age (data drift / bug)
user_profiles[3]["age"] = 200

# Build daily activity
daily_activity = []
for u in user_profiles:
    base_steps = int(rng.normal(7500, 2000))
    for d in range(N_DAYS):
        date = start_date + timedelta(days=d)
        steps   = max(0, int(rng.normal(base_steps, 1500)))
        dist_km = round(steps * 0.00075, 2)
        daily_activity.append({
            "user_id":        u["user_id"],
            "date":           date.isoformat(),
            "steps":          steps,
            "distance_km":    dist_km,
            "calories":       int(1600 + steps * 0.04 + rng.normal(0, 100)),
            "active_minutes": int(max(0, rng.normal(35, 15))),
            "sedentary_min":  int(max(0, rng.normal(700, 90))),
        })

# Inject some bad rows
daily_activity[100]["steps"]    = -500           # impossible
daily_activity[200]["steps"]    = 250_000        # impossible (≈190 miles)
daily_activity[300]["distance_km"] = None
daily_activity[400]["steps"]    = None

# Build sleep records (one per user-night)
sleep_records = []
for u in user_profiles:
    for d in range(N_DAYS):
        sleep_start = start_date + timedelta(days=d, hours=22, minutes=int(rng.integers(0, 60)))
        total       = int(rng.normal(420, 60))           # ≈ 7 h
        rem         = int(total * rng.uniform(0.15, 0.25))
        deep        = int(total * rng.uniform(0.10, 0.20))
        light       = total - rem - deep - int(total * 0.10)
        sleep_records.append({
            "record_id":      fake.uuid4(),
            "user_id":        u["user_id"],
            "sleep_start":    sleep_start.isoformat(),
            "sleep_end":      (sleep_start + timedelta(minutes=total)).isoformat(),
            "total_minutes":  total,
            "minutes_rem":    rem,
            "minutes_deep":   deep,
            "minutes_light":  max(light, 0),
            "minutes_awake":  int(total * 0.10),
            "efficiency_pct": round(rng.uniform(0.80, 0.95) * 100, 1),
        })

# Inject duplicates that simulate the iOS background-fetch bug
for idx in [10, 11, 12, 13, 14]:
    dup = sleep_records[idx].copy()
    dup["record_id"] = fake.uuid4()           # different ID, same logical sleep
    sleep_records.append(dup)

# Inject impossible sleep durations
sleep_records[500]["total_minutes"] = 1700     # > 24 h
sleep_records[600]["total_minutes"] = -30      # negative

# ---- Write to TinyDB ------------------------------------------------------
if TINYDB_PATH.exists():
    TINYDB_PATH.unlink()
tdb = TinyDB(TINYDB_PATH)
tdb.table("user_profiles").insert_multiple(user_profiles)
tdb.table("daily_activity").insert_multiple(daily_activity)
tdb.table("sleep_records").insert_multiple(sleep_records)

log.info("Loaded %d user profiles into TinyDB", len(user_profiles))
log.info("Loaded %d daily_activity docs", len(daily_activity))
log.info("Loaded %d sleep_records docs (incl. %d duplicates)",
         len(sleep_records), 5)


### 1c. Extract from both stores — the analyst's daily query


In [ ]:
# ---------------------------------------------------------------------------
# This is what would normally use psycopg2 + pymongo in production.
# Same logical query — we just swap in SQLite / TinyDB locally.
# ---------------------------------------------------------------------------

# --- From the "Redshift" warehouse (SQLite) --------------------------------
def fetch_cardio_with_demographics(min_age: int = 18) -> pd.DataFrame:
    sql = '''
        SELECT  c.seqn, c.exam_date, c.systolic_bp, c.diastolic_bp,
                c.pulse_rate_bpm, d.age_years, d.gender
          FROM  cardiovascular c
          JOIN  demographics  d ON d.seqn = c.seqn
         WHERE  d.age_years >= ?
    '''
    with sqlite3.connect(SQLITE_PATH) as conn:
        return pd.read_sql(sql, conn, params=(min_age,))

cardio_join = fetch_cardio_with_demographics(min_age=18)
print(f"Pulled {len(cardio_join):,} cardio+demographics rows from clinical warehouse")
cardio_join.head()


In [ ]:
# --- From the "MongoDB" telemetry store (TinyDB) ---------------------------
def fetch_daily_activity_df() -> pd.DataFrame:
    docs = tdb.table("daily_activity").all()
    df = pd.DataFrame(docs)
    df["date"] = pd.to_datetime(df["date"])
    return df

def fetch_sleep_df() -> pd.DataFrame:
    docs = tdb.table("sleep_records").all()
    df = pd.DataFrame(docs)
    df["sleep_start"] = pd.to_datetime(df["sleep_start"])
    df["sleep_end"]   = pd.to_datetime(df["sleep_end"])
    return df

def fetch_user_profiles_df() -> pd.DataFrame:
    return pd.DataFrame(tdb.table("user_profiles").all())

activity_df = fetch_daily_activity_df()
sleep_df    = fetch_sleep_df()
profiles_df = fetch_user_profiles_df()

print(f"daily_activity:  {len(activity_df):,} rows")
print(f"sleep_records:   {len(sleep_df):,} rows")
print(f"user_profiles:   {len(profiles_df):,} rows")
profiles_df.head(3)


---
## 🟩 Phase 2 — Data Governance & PII Masking

A **data governance framework** has two faces:

1. **Policy** (the rules) — documented in plain English so engineers, lawyers, and clinicians can all agree.
2. **Enforcement** (the code) — functions that mechanically strip / mask sensitive fields before data reaches the analytics layer.

We implement both below. Key rules:

| Rule | Implementation |
|---|---|
| **Direct PII** (name, email, phone, device ID) | Replaced with salted SHA-256 hash (`pseudonymize`) |
| **Email domain** preserved for analytics | `mask_email("alice@x.com") → "*lic*@x.com"` |
| **Age** capped at 89 (HIPAA Safe Harbor) | `generalize_age` |
| **ZIP** truncated to 3 digits, blanked for low-pop ZIPs | `generalize_zip` |
| **Timestamps** standardized to UTC | `pd.to_datetime(..., utc=True)` |
| **RBAC** four-tier role matrix | `role_can_read(role, sensitivity)` |


In [ ]:
# ---------------------------------------------------------------------------
# Governance layer — kept identical to src/governance/anonymizer.py so this
# notebook is a faithful preview of the production module.
# ---------------------------------------------------------------------------
import hashlib, re

_SALT = "wellness-engine-v1::change-in-prod"

PII_DIRECT = {"full_name", "name", "email", "phone", "phone_number",
              "ssn", "device_id", "imei", "mac_address"}
PII_INDIRECT = {"zip", "zip_code", "postal_code",
                "birth_date", "date_of_birth", "dob",
                "age", "age_years"}

def pseudonymize(v) -> str:
    if pd.isna(v): return ""
    return hashlib.sha256(f"{_SALT}::{v}".encode()).hexdigest()[:16]

def mask_email(v: str) -> str:
    if not isinstance(v, str) or "@" not in v: return ""
    local, domain = v.split("@", 1)
    masked = "*" * len(local) if len(local) <= 2 \
             else local[0] + "*" * (len(local) - 2) + local[-1]
    return f"{masked}@{domain}"

def generalize_zip(z) -> str:
    digits = re.sub(r"\D", "", str(z))[:3]
    forbidden = {"036","059","063","102","203","556","692","790","821",
                 "823","830","831","878","879","884","890","893"}
    return "000" if digits in forbidden else digits.ljust(3, "0")

def generalize_age(a) -> int:
    if pd.isna(a): return -1
    return min(int(a), 89)

def anonymize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        low = col.lower()
        if low in PII_DIRECT:
            out[col] = out[col].astype(str).map(
                mask_email if "email" in low else pseudonymize)
        elif low in PII_INDIRECT:
            if "zip" in low:   out[col] = out[col].astype(str).map(generalize_zip)
            elif "age" in low: out[col] = out[col].map(generalize_age)
    return out

# RBAC matrix
def role_can_read(role: str, sensitivity: str) -> bool:
    matrix = {
        "admin":    {"public","internal","sensitive","restricted"},
        "engineer": {"public","internal","sensitive"},
        "analyst":  {"public","internal"},
        "intern":   {"public"},
    }
    return sensitivity in matrix.get(role, set())

print("Governance functions loaded.\n")
print("Quick RBAC demo:")
for role in ["admin","engineer","analyst","intern"]:
    print(f"  {role:10s} → can read sensitive? {role_can_read(role,'sensitive')}")


In [ ]:
# ---------------------------------------------------------------------------
# Apply the governance pass to the wearable user profiles.
# Compare BEFORE and AFTER side-by-side.
# ---------------------------------------------------------------------------
print("BEFORE (raw — contains PII):")
display(profiles_df[["user_id","full_name","email","phone","device_id","age"]].head(3))

profiles_clean = anonymize_dataframe(profiles_df)

print("\nAFTER (governance pass applied):")
display(profiles_clean[["user_id","full_name","email","phone","device_id","age"]].head(3))

# Persist the cleaned version — only this can go to the analytics warehouse
profiles_clean.to_parquet(WORKDIR / "user_profiles_clean.parquet", index=False)
log.info("Wrote cleaned profiles → user_profiles_clean.parquet")


---
## 🟧 Phase 3 — Quality KPIs, Logic Rules & Statistical Anomaly Detection

This phase is the heart of the project. It runs **three layers** of checks:

1. **Tracking KPIs** — `completeness_rate`, `uniqueness`, `validity`, `timeliness`
2. **Logic-based rules** — physically impossible records (negative HR, age > 120, systolic ≤ diastolic, …)
3. **Statistical anomaly detection** — Z-score, IQR, and modified Z-score (MAD) outlier flags, plus **group-wise Z-scores** for context-aware detection (e.g. blood pressure normalized within age bracket)

Plus **statistical imputation** for missing data (`SimpleImputer` for univariate, `KNNImputer` for correlated wearable time-series).


### 3a. Tracking KPIs — completeness, uniqueness, validity


In [ ]:
# ---------------------------------------------------------------------------
# Data Quality KPI calculator — what you'd run as a scheduled cron / Airflow DAG.
# ---------------------------------------------------------------------------
def completeness_rate(df: pd.DataFrame, cols: list[str] | None = None) -> dict:
    cols = cols or df.columns.tolist()
    return {c: round(1 - df[c].isna().mean(), 4) for c in cols if c in df.columns}

def uniqueness_rate(df: pd.DataFrame, keys: list[str]) -> float:
    if not all(k in df.columns for k in keys):
        return float("nan")
    return round(1 - df.duplicated(subset=keys).mean(), 4)

def validity_rate(df: pd.DataFrame, column: str, lo: float, hi: float) -> float:
    s = pd.to_numeric(df[column], errors="coerce")
    valid = s.between(lo, hi)
    return round(valid.sum() / max(len(s), 1), 4)

CLINICAL_BOUNDS = {
    "age_years":     (0, 120),
    "systolic_bp":   (50, 260),
    "diastolic_bp":  (30, 160),
    "pulse_rate_bpm":(30, 220),
    "steps":         (0, 100_000),
    "total_minutes": (0, 1440),
    "hba1c":         (3.0, 20.0),
    "fasting_glucose":(30, 600),
}

# ---- Compute the KPIs across every table ----------------------------------
demo_df = pd.read_sql("SELECT * FROM demographics", sqlite3.connect(SQLITE_PATH))
card_df = pd.read_sql("SELECT * FROM cardiovascular", sqlite3.connect(SQLITE_PATH))
labs_df = pd.read_sql("SELECT * FROM lab_results",   sqlite3.connect(SQLITE_PATH))

kpi_rows = []
for tbl_name, df, keys in [
    ("demographics",   demo_df, ["seqn"]),
    ("cardiovascular", card_df, ["seqn", "exam_date"]),
    ("lab_results",    labs_df, ["seqn", "sample_date"]),
    ("daily_activity", activity_df, ["user_id", "date"]),
    ("sleep_records",  sleep_df,    ["user_id", "sleep_start"]),
]:
    comp = completeness_rate(df)
    kpi_rows.append({
        "table":              tbl_name,
        "rows":               len(df),
        "avg_completeness":   round(np.mean(list(comp.values())), 4),
        "uniqueness":         uniqueness_rate(df, keys),
        "n_columns":          len(df.columns),
    })

kpi_df = pd.DataFrame(kpi_rows)
display(kpi_df)


### 3b. Logic-based rules — physically impossible values


In [ ]:
# ---------------------------------------------------------------------------
# Run hard rules and write every failure to data_quality_log.
# This mirrors src/quality_engine/logic_rules.py.
# ---------------------------------------------------------------------------
from dataclasses import dataclass, field

@dataclass
class QualityReport:
    check_name: str
    table: str
    total_rows: int
    failed_rows: int
    failed_index: list = field(default_factory=list)
    description: str = ""
    @property
    def pass_rate(self): return 1.0 if self.total_rows == 0 else 1 - self.failed_rows / self.total_rows

def check_range(df, column, table_name):
    if column not in df.columns or column not in CLINICAL_BOUNDS:
        return None
    lo, hi = CLINICAL_BOUNDS[column]
    s = pd.to_numeric(df[column], errors="coerce")
    mask = (s < lo) | (s > hi)
    return QualityReport(f"range::{column}", table_name, len(df),
                         int(mask.sum()), df.index[mask].tolist(),
                         f"{column} must be in [{lo},{hi}]")

def check_logical_consistency(df, table_name):
    bad = pd.Series(False, index=df.index)
    notes = []
    if {"systolic_bp","diastolic_bp"}.issubset(df.columns):
        b = df["systolic_bp"] <= df["diastolic_bp"]
        bad |= b.fillna(False); notes.append("systolic ≤ diastolic" if b.any() else "")
    return QualityReport("logical_consistency", table_name, len(df),
                         int(bad.sum()), df.index[bad].tolist(),
                         "; ".join(n for n in notes if n))

def check_duplicates(df, keys, table_name):
    if not all(k in df.columns for k in keys): return None
    mask = df.duplicated(subset=keys, keep=False)
    return QualityReport(f"uniqueness::{'+'.join(keys)}", table_name, len(df),
                         int(mask.sum()), df.index[mask].tolist(),
                         f"composite key must be unique: {keys}")

# Run every applicable check
all_reports = []
for r in [
    check_range(demo_df, "age_years", "demographics"),
    check_range(card_df, "systolic_bp", "cardiovascular"),
    check_range(card_df, "diastolic_bp", "cardiovascular"),
    check_range(card_df, "pulse_rate_bpm", "cardiovascular"),
    check_range(labs_df, "fasting_glucose", "lab_results"),
    check_range(labs_df, "hba1c", "lab_results"),
    check_range(activity_df, "steps", "daily_activity"),
    check_range(sleep_df, "total_minutes", "sleep_records"),
    check_logical_consistency(card_df, "cardiovascular"),
    check_duplicates(sleep_df, ["user_id","sleep_start"], "sleep_records"),
]:
    if r is not None:
        all_reports.append(r)

# Tidy summary
quality_summary = pd.DataFrame([{
    "check":       r.check_name,
    "table":       r.table,
    "total_rows":  r.total_rows,
    "failed_rows": r.failed_rows,
    "pass_rate":   round(r.pass_rate, 4),
    "description": r.description,
} for r in all_reports]).sort_values("pass_rate")

print("📋 Data Quality Check Summary")
display(quality_summary)


In [ ]:
# ---------------------------------------------------------------------------
# Persist every failed check to data_quality_log — the audit trail used in
# Phase 4 root-cause investigation.
# ---------------------------------------------------------------------------
with sqlite3.connect(SQLITE_PATH) as conn:
    cur = conn.cursor()
    now = datetime.utcnow().isoformat()
    for r in all_reports:
        for idx in r.failed_index[:200]:   # cap to keep log readable
            cur.execute('''
                INSERT INTO data_quality_log
                   (check_name, table_name, column_name, record_id,
                    severity, description, flagged_value, detected_at)
                VALUES (?,?,?,?,?,?,?,?)''',
                (r.check_name, r.table, r.check_name.split("::")[-1],
                 str(idx), "WARN", r.description, "<see source row>", now))
    conn.commit()
    n_logged = pd.read_sql("SELECT COUNT(*) AS n FROM data_quality_log", conn).iloc[0,0]
print(f"📒 Wrote {n_logged:,} rows to data_quality_log")


### 3c. Statistical anomaly detection — Z-score, IQR, modified Z-score


In [ ]:
# ---------------------------------------------------------------------------
# Apply three outlier detectors to every numeric column.
# Recommended reading: Iglewicz & Hoaglin (1993) for the modified Z-score.
# ---------------------------------------------------------------------------
def zscore_outliers(s, threshold=3.0):
    s = pd.to_numeric(s, errors="coerce")
    mu = s.mean()
    sd = s.std(ddof=0)
    if sd == 0 or pd.isna(sd):
        return pd.Series(False, index=s.index), float("nan"), float("nan")
    z = ((s - mu) / sd).abs()
    mask = (z > threshold).fillna(False)
    return mask, float(mu - threshold * sd), float(mu + threshold * sd)

def iqr_outliers(s, k=1.5):
    s = pd.to_numeric(s, errors="coerce")
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - k*iqr, q3 + k*iqr
    return (s < lo) | (s > hi), float(lo), float(hi)

def modified_zscore_outliers(s, threshold=3.5):
    s = pd.to_numeric(s, errors="coerce")
    med = s.median(); mad = (s - med).abs().median()
    if mad == 0: return pd.Series(False, index=s.index), float("nan"), float("nan")
    mz = 0.6745 * (s - med) / mad
    return mz.abs() > threshold, float(med - threshold*mad/0.6745), float(med + threshold*mad/0.6745)

# Run on the cardio table
audit_rows = []
for col in ["systolic_bp", "diastolic_bp", "pulse_rate_bpm"]:
    for fn, name in [(zscore_outliers, "zscore"),
                     (iqr_outliers, "iqr"),
                     (modified_zscore_outliers, "modified_zscore")]:
        mask, lo, hi = fn(card_df[col])
        audit_rows.append({
            "column": col, "method": name,
            "n_outliers": int(mask.sum()),
            "outlier_pct": round(100*mask.sum()/len(card_df), 2),
            "lower_bound": round(lo, 2), "upper_bound": round(hi, 2),
        })
audit_df = pd.DataFrame(audit_rows)
print("📊 Statistical Outlier Audit — Cardiovascular Table")
display(audit_df)


In [ ]:
# ---------------------------------------------------------------------------
# Group-wise Z-scores: blood pressure normalized within age bracket.
# This is exactly what the project brief asks for.
# ---------------------------------------------------------------------------
joined = cardio_join.copy()
joined["age_bracket"] = pd.cut(joined["age_years"],
                               bins=[0,30,45,60,75,200],
                               labels=["18-30","31-45","46-60","61-75","76+"])

def z_by_group(df, value_col, group_col):
    return df.groupby(group_col, observed=True)[value_col].transform(
        lambda x: (x - x.mean()) / (x.std() or 1)
    )

joined["bp_zscore_in_bracket"] = z_by_group(joined, "systolic_bp", "age_bracket")
group_outliers = joined.loc[joined["bp_zscore_in_bracket"].abs() > 3]

print(f"Group-wise outliers (|Z|>3 within age bracket): {len(group_outliers)}")
display(group_outliers[["seqn","age_years","age_bracket","systolic_bp","bp_zscore_in_bracket"]]
        .sort_values("bp_zscore_in_bracket", key=abs, ascending=False).head(10))


### 3d. Statistical imputation — fill the gaps responsibly


In [ ]:
# ---------------------------------------------------------------------------
# Two imputation strategies:
#   • SimpleImputer    — fast median fill for univariate gaps
#   • KNNImputer       — multivariate, uses correlated columns (best for
#                        wearable time-series where steps ↔ sleep ↔ HR)
# ---------------------------------------------------------------------------
print("BEFORE imputation — null counts in lab panel:")
display(labs_df[["total_cholesterol","hdl_cholesterol","ldl_cholesterol",
                 "fasting_glucose","hba1c"]].isna().sum())

# --- 1. SimpleImputer (median) ---------------------------------------------
labs_simple = labs_df.copy()
imp = SimpleImputer(strategy="median")
cols = ["total_cholesterol","hdl_cholesterol","ldl_cholesterol",
        "fasting_glucose","hba1c"]
labs_simple[cols] = imp.fit_transform(labs_simple[cols])

# --- 2. KNN imputation on wearable time-series -----------------------------
activity_for_knn = activity_df[["steps","distance_km","calories",
                                "active_minutes","sedentary_min"]].copy()
knn = KNNImputer(n_neighbors=5, weights="distance")
activity_imputed = pd.DataFrame(
    knn.fit_transform(activity_for_knn),
    columns=activity_for_knn.columns,
    index=activity_for_knn.index,
)

print("\nAFTER imputation — labs (median):")
display(labs_simple[cols].isna().sum())
print("\nAFTER imputation — wearable activity (KNN):")
display(activity_imputed.isna().sum())


---
## 🟪 Phase 4 — Root-Cause Investigation & Cross-Functional Collaboration

Quality checks tell us **something is wrong**. Now we play data detective and figure out **why**, then translate the finding into terms each team can act on.

The investigation pattern:

1. **Reproduce** the anomaly from `data_quality_log`
2. **Trace** the lineage backwards to the source records
3. **Categorize** the root cause: device bug, user error, threshold misconfiguration, or clinical reality
4. **Communicate** the finding to the right audience (Engineering, Product, Operations)


In [ ]:
# ---------------------------------------------------------------------------
# Investigation 1: the duplicate sleep records
# ---------------------------------------------------------------------------
print("🔎 INVESTIGATION 1 — Duplicate sleep records\n")

dup_mask = sleep_df.duplicated(subset=["user_id","sleep_start"], keep=False)
dup_rows = sleep_df.loc[dup_mask].sort_values(["user_id","sleep_start"])

print(f"Total duplicates: {dup_mask.sum()} rows across "
      f"{dup_rows[['user_id','sleep_start']].drop_duplicates().shape[0]} unique nights\n")

display(dup_rows[["record_id","user_id","sleep_start","total_minutes"]].head(10))

print("\n→ Root cause hypothesis: the API/device retry logic is reposting "
      "the same sleep payload with a fresh `record_id`.")
print("→ Owner: Engineering")
print("→ Fix: enforce upsert on (user_id, sleep_start) at the API layer.")


In [ ]:
# ---------------------------------------------------------------------------
# Investigation 2: impossible cardio values — clinical vs device bug?
# ---------------------------------------------------------------------------
print("🔎 INVESTIGATION 2 — Out-of-range cardiovascular records\n")

bad_cardio = card_df.loc[
    ~card_df["systolic_bp"].between(50, 260) |
    ~card_df["diastolic_bp"].between(30, 160) |
    ~card_df["pulse_rate_bpm"].between(30, 220)
].copy()

bad_cardio["likely_cause"] = np.where(
    (bad_cardio["systolic_bp"].abs() > 500) |
    (bad_cardio["pulse_rate_bpm"].abs() > 300) |
    (bad_cardio["systolic_bp"] < 0),
    "device error", "possible medical event")

display(bad_cardio[["seqn","systolic_bp","diastolic_bp","pulse_rate_bpm","likely_cause"]])

print("\n→ 'device error' rows: send to Engineering for ingestion-layer fix.")
print("→ 'possible medical event' rows: forward (anonymized) to Clinical Ops.")


In [ ]:

# ---------------------------------------------------------------------------
# Cross-functional briefings — three audiences, one underlying truth.
# ---------------------------------------------------------------------------
DUP_PCT = 100 * 10 / len(sleep_df)
N_CLINICAL = bad_cardio[bad_cardio.likely_cause == 'possible medical event'].shape[0]

engineering_msg = [
    "[DATA-QUALITY] Duplicate sleep records — likely iOS sync retry bug",
    "• 5 unique nights × 2 record_ids each = 10 duplicated rows",
    "• All share identical (user_id, sleep_start) but distinct record_id",
    "• Suggested fix: upsert on (user_id, sleep_start) in the API layer;",
    "  add idempotency token in iOS client.",
    "• Log query: SELECT * FROM data_quality_log",
    "              WHERE check_name = 'uniqueness::user_id+sleep_start';",
]

product_msg = [
    f"TL;DR — Sleep data is currently inflated by ~{DUP_PCT:.2f}% due to",
    "duplicate records (root cause: iOS background-fetch bug, see Eng note).",
    "After the fix lands, the dashboard will appear to drop — that's a CORRECTION,",
    "not a regression. Communicate this in advance.",
    "",
    "Recommended next experiment: A/B test a 'wind-down reminder' on users",
    "whose median sleep < 7 h, measured against `total_minutes`.",
]

clinical_msg = [
    f"We have {N_CLINICAL} record(s) where systolic BP exceeds 260 mmHg or sits",
    "inside hypertensive-crisis territory but otherwise looks plausible (no",
    "negative HR, no impossible diastolic). These cannot be auto-corrected.",

    "",
    "Recommended action: forward the pseudonymized seqn list to your team for",
    "manual review. We have not flagged the user identities.",
]

def banner(title):
    print("=" * 70)
    print(title)
    print("=" * 70)

banner("📩 FOR ENGINEERING (Slack-style)")
print("\n".join(engineering_msg))
print()
banner("📊 FOR PRODUCT MANAGER (briefing-style)")
print("\n".join(product_msg))
print()
banner("🩺 FOR CLINICAL OPERATIONS (memo-style)")
print("\n".join(clinical_msg))


---
## 🟫 Phase 5 — Customized Reports & Dashboards

We build **two distinct dashboards**, each calibrated to its audience:

| Dashboard | Audience | Charts |
|---|---|---|
| **Executive / Wellness Dashboard** | Product + Leadership | DAU, average sleep, BP distribution, wellness index distribution |
| **Data Health Dashboard** | Engineering + Data Ops | DQS over time, failed checks by table, outlier rates by method |

Both are rendered with Plotly, so they're interactive inside the notebook and exportable to HTML.


### 5a. Executive / Wellness Dashboard


In [ ]:
# ---------------------------------------------------------------------------
# Clean data (post-governance + post-quality) for the executive view
# ---------------------------------------------------------------------------
clean_activity = activity_df.dropna(subset=["steps"]).copy()
clean_activity = clean_activity[clean_activity["steps"].between(0, 100_000)]
clean_activity["week"] = pd.to_datetime(clean_activity["date"]).dt.to_period("W").dt.start_time

clean_cardio = card_df[
    card_df["systolic_bp"].between(50, 260) &
    card_df["diastolic_bp"].between(30, 160) &
    (card_df["systolic_bp"] > card_df["diastolic_bp"])
].merge(demo_df[["seqn","age_years"]], on="seqn")

# Construct a simple "Wellness Index" 0–100
clean_sleep = sleep_df[sleep_df["total_minutes"].between(0, 1440)].copy()
sleep_score = (clean_sleep["total_minutes"] / 480).clip(0, 1) * 100
activity_score = (clean_activity["steps"] / 10000).clip(0, 1) * 100

# --- Build the 2x2 dashboard -----------------------------------------------
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Weekly Active Users (DAU rolled up)",
        "Sleep Duration Distribution (minutes)",
        "Blood Pressure by Age",
        "Wellness Index Distribution",
    ),
    specs=[[{"type":"scatter"},{"type":"histogram"}],
           [{"type":"scatter"},{"type":"histogram"}]],
)

# Top-left: weekly users
wau = clean_activity.groupby("week")["user_id"].nunique().reset_index()
fig.add_trace(go.Scatter(x=wau["week"], y=wau["user_id"],
                         mode="lines+markers", name="WAU",
                         line=dict(color="#1f77b4")), row=1, col=1)

# Top-right: sleep distribution
fig.add_trace(go.Histogram(x=clean_sleep["total_minutes"], nbinsx=40,
                           marker_color="#9467bd", name="Sleep min"),
              row=1, col=2)

# Bottom-left: BP scatter coloured by age
fig.add_trace(go.Scatter(
    x=clean_cardio["age_years"], y=clean_cardio["systolic_bp"],
    mode="markers", marker=dict(color=clean_cardio["age_years"],
    colorscale="Viridis", size=5, opacity=0.5,
    colorbar=dict(title="Age", x=0.45)),
    name="Systolic"), row=2, col=1)

# Bottom-right: Wellness Index distribution
wi = pd.concat([sleep_score, activity_score]).reset_index(drop=True)
fig.add_trace(go.Histogram(x=wi, nbinsx=30, marker_color="#2ca02c",
                           name="Wellness Idx"), row=2, col=2)

fig.update_layout(
    title=dict(text="🩺 Wellness Platform — Executive Dashboard",
               font=dict(size=20)),
    height=750, showlegend=False,
    template="plotly_white",
)
fig.update_xaxes(title_text="Week",          row=1, col=1)
fig.update_xaxes(title_text="Minutes",       row=1, col=2)
fig.update_xaxes(title_text="Age (years)",   row=2, col=1)
fig.update_xaxes(title_text="Wellness Index",row=2, col=2)
fig.update_yaxes(title_text="Unique users",  row=1, col=1)
fig.update_yaxes(title_text="Count",         row=1, col=2)
fig.update_yaxes(title_text="Systolic BP",   row=2, col=1)
fig.update_yaxes(title_text="Count",         row=2, col=2)

fig.show()
fig.write_html(WORKDIR / "executive_dashboard.html")
print(f"\n💾 Saved → {WORKDIR / 'executive_dashboard.html'}")


### 5b. Data Health Dashboard — for the Engineering team


In [ ]:
# ---------------------------------------------------------------------------
# Build the data health dashboard from the quality_summary + audit_df.
# ---------------------------------------------------------------------------
fig2 = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Pass-rate by Quality Check",
        "Failed Rows by Table",
        "Outlier Counts — Cardio (by Method)",
        "Completeness by Column (Top 10 Worst)",
    ),
    specs=[[{"type":"bar"},{"type":"bar"}],
           [{"type":"bar"},{"type":"bar"}]],
)

# Top-left: pass rate per check (sorted)
qs = quality_summary.sort_values("pass_rate")
fig2.add_trace(go.Bar(
    x=qs["pass_rate"], y=qs["check"], orientation="h",
    marker_color=np.where(qs["pass_rate"] < 0.99, "crimson", "seagreen"),
    text=[f"{p:.2%}" for p in qs["pass_rate"]], textposition="outside",
), row=1, col=1)

# Top-right: failed rows by table
ft = quality_summary.groupby("table")["failed_rows"].sum().reset_index()
fig2.add_trace(go.Bar(x=ft["table"], y=ft["failed_rows"],
                      marker_color="#ff7f0e",
                      text=ft["failed_rows"], textposition="outside"),
               row=1, col=2)

# Bottom-left: outlier method comparison
fig2.add_trace(go.Bar(
    x=audit_df["column"] + " / " + audit_df["method"],
    y=audit_df["n_outliers"],
    marker_color="#9467bd",
    text=audit_df["n_outliers"], textposition="outside",
), row=2, col=1)

# Bottom-right: lowest completeness columns
all_completeness = {}
for tbl_name, df in [("demo", demo_df), ("cardio", card_df),
                     ("labs", labs_df), ("activity", activity_df)]:
    for c, v in completeness_rate(df).items():
        all_completeness[f"{tbl_name}.{c}"] = v
comp_df = pd.Series(all_completeness).sort_values().head(10).reset_index()
comp_df.columns = ["column", "completeness"]
fig2.add_trace(go.Bar(x=comp_df["completeness"], y=comp_df["column"],
                      orientation="h", marker_color="#d62728",
                      text=[f"{v:.2%}" for v in comp_df["completeness"]],
                      textposition="outside"),
               row=2, col=2)

fig2.update_layout(
    title=dict(text="🔧 Data Health Dashboard (Engineering View)",
               font=dict(size=20)),
    height=850, showlegend=False, template="plotly_white",
)
fig2.update_xaxes(title_text="Pass rate",   row=1, col=1, range=[0.85, 1.01])
fig2.update_xaxes(title_text="Table",       row=1, col=2)
fig2.update_xaxes(title_text="Column / method", row=2, col=1, tickangle=45)
fig2.update_xaxes(title_text="Completeness",row=2, col=2, range=[0, 1.05])

fig2.show()
fig2.write_html(WORKDIR / "data_health_dashboard.html")
print(f"\n💾 Saved → {WORKDIR / 'data_health_dashboard.html'}")


---
## ✅ Summary — What This Notebook Delivered

| Responsibility | Where in the notebook |
|---|---|
| Implement a data governance framework | Phase 2 (`anonymize_dataframe`, RBAC, masking) |
| Track and assess data quality | Phase 3a (completeness, uniqueness, validity KPIs) |
| Automated checks for inconsistencies | Phase 3b (range, logical, duplicate checks → `data_quality_log`) |
| Statistical techniques | Phase 3c–d (Z-score, IQR, modified Z, group-wise Z, KNN imputation) |
| Customized reports and dashboards | Phase 5 (executive + data-health Plotly dashboards) |
| Extract from databases (Redshift, MongoDB) | Phase 1c (SQL joins + document queries) |
| Identify and resolve root causes | Phase 4 (duplicate sleep + bad cardio investigations) |
| Cross-functional collaboration | Phase 4 (Engineering / Product / Clinical briefings) |
| Create and maintain documentation | `docs/data_dictionary.md` + `docs/governance_policy.md` |

### 🚀 Next steps for production

- Containerize: `docker compose up -d` brings the same flow to real Postgres + MongoDB
- Schedule: wire the checks into Airflow (`@daily`)
- Alert: when DQS drops below a threshold, page the on-call engineer
- Iterate: review thresholds with Clinical Ops monthly

### 🔗 GitHub repository structure

```
wellness-data-integrity-engine/
├── infrastructure/   docker-compose, init scripts
├── src/              extraction, governance, quality_engine
├── notebooks/        this notebook + investigation + dashboards
├── docs/             data dictionary, governance policy, briefings
└── tests/            pytest suite (20 tests, all green)
```
